<a href="https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/02_structured_outputs_three_ways.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" style="height: 28px; vertical-align: middle;"/></a>  **[▶️ Run this notebook in Colab](https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/02_structured_outputs_three_ways.ipynb)**

# Structured outputs - three ways

> **Track 04 - Agents · Notebook 02 · Runtime: ≈30 s on CPU**
>
> **Prerequisites:** `04_agents/01` (ReAct from scratch).
>
> **References:**
> - [Outlines](https://github.com/dottxt-ai/outlines) for FSM-constrained generation.
> - Willard & Louf 2023, *Efficient Guided Generation for LLMs*
>   ([2307.09702](https://arxiv.org/abs/2307.09702)).
> - [Pydantic](https://docs.pydantic.dev/).

---

## What

LLM code that needs to parse the model's output into a schema -
tool-calling, function-calling, structured RAG answers - has three
common strategies:

1. **Prompt-only.** Ask the model to emit JSON, parse it. Simple,
   works for strong models, breaks occasionally.
2. **Validate + retry.** Parse with Pydantic; on validation failure,
   send the error back to the model and ask it to fix. Higher
   compliance; extra latency on failures.
3. **FSM-constrained** (Outlines / jsonschema-guided decoding). Modify
   sampling so only tokens that satisfy the grammar can be emitted.
   100 % schema compliance by construction; latency competitive
   with unconstrained.

We simulate a flaky "LLM" that produces valid JSON 70 % of the time
and malformed text 30 % of the time. Each strategy is run against the
same 200 requests; we measure validity (parses at all) and compliance
(matches the schema).


In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import numpy as np
from pydantic import BaseModel, Field, ValidationError

from llm_systems_cookbook._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_02_structured_outputs_three_ways")


## The schema

We want a `Person` object with a name, age in [0, 130], and a list of
hobbies. Pydantic gives us both validation and a JSON schema for the
FSM-constrained path.


In [ ]:
class Person(BaseModel):
    name: str = Field(min_length=1, max_length=50)
    age: int = Field(ge=0, le=130)
    hobbies: list[str] = Field(default_factory=list, max_length=10)


SCHEMA = Person.model_json_schema()
print("JSON schema top-level keys:", list(SCHEMA.keys()))
print(f"required fields: {SCHEMA.get('required', [])}")


## Flaky LLM stub

Three failure modes matching what real LLMs actually emit:

- **Valid JSON** matching schema (70 %).
- **Valid JSON** but violating schema - age negative or >130, name
  empty (15 %).
- **Invalid JSON** - trailing comma, missing quote, pre/post-amble
  text (15 %).

The `LLM.generate` function returns a string. Strategies below have
to figure out what to do with it.


In [ ]:
set_seed(0)
rng = np.random.default_rng(0)

GOOD_NAMES = ["Alice", "Bob", "Carol", "Dan", "Eve"]
GOOD_HOBBIES = [["chess"], ["hiking", "painting"], [], ["cooking"]]


class LLM:
    '''Deterministic stub with mixed output quality.'''

    def __init__(self) -> None:
        self.rng = np.random.default_rng(0)

    def generate(self, prompt: str, retry_correction: str | None = None) -> str:
        # On retry, always produce valid output (simulating that the model
        # uses the error message effectively).
        if retry_correction is not None:
            name = self.rng.choice(GOOD_NAMES)
            age = int(self.rng.integers(10, 80))
            hobbies = GOOD_HOBBIES[int(self.rng.integers(0, len(GOOD_HOBBIES)))]
            return json.dumps({"name": str(name), "age": age, "hobbies": list(hobbies)})

        r = self.rng.random()
        if r < 0.70:
            name = self.rng.choice(GOOD_NAMES)
            age = int(self.rng.integers(10, 80))
            hobbies = GOOD_HOBBIES[int(self.rng.integers(0, len(GOOD_HOBBIES)))]
            return json.dumps({"name": str(name), "age": age, "hobbies": list(hobbies)})
        if r < 0.85:
            # Valid JSON, schema-violating.
            name = self.rng.choice(GOOD_NAMES)
            bad_age = int(self.rng.choice([-5, 200, 999]))
            return json.dumps({"name": str(name), "age": bad_age, "hobbies": []})
        # Malformed JSON: pre-amble, trailing comma.
        name = self.rng.choice(GOOD_NAMES)
        return f"Sure! Here is the object: {{\"name\": \"{name}\", \"age\": 42,}}"


def extract_first_json(text: str) -> str | None:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    return text[start : end + 1]


## Three strategies

Each takes an `LLM` instance and a prompt and returns either a
`Person` or `None`.


In [ ]:
def strategy_prompt_only(llm: LLM, prompt: str) -> Person | None:
    raw = llm.generate(prompt)
    chunk = extract_first_json(raw)
    if chunk is None:
        return None
    try:
        data = json.loads(chunk)
        return Person(**data)
    except (json.JSONDecodeError, ValidationError, TypeError):
        return None


def strategy_validate_retry(llm: LLM, prompt: str, max_retries: int = 2) -> Person | None:
    correction: str | None = None
    for _ in range(max_retries + 1):
        raw = llm.generate(prompt, retry_correction=correction)
        chunk = extract_first_json(raw) or raw
        try:
            data = json.loads(chunk)
        except json.JSONDecodeError as e:
            correction = f"Your output failed JSON parsing: {e.msg}. Output ONLY valid JSON."
            continue
        try:
            return Person(**data)
        except (ValidationError, TypeError) as e:
            correction = f"Your output failed schema validation: {e}. Fix the fields and re-emit."
            continue
    return None


def strategy_fsm_constrained(llm: LLM, prompt: str) -> Person:
    '''Simulated FSM-constrained decoding.

    In production Outlines (or similar) forces the sampler to only
    produce tokens that satisfy the JSON schema. Here we simulate with
    a generator that *always* returns valid, schema-compliant output.
    '''
    # The FSM guarantees validity.
    name = rng.choice(GOOD_NAMES)
    age = int(rng.integers(10, 80))
    hobbies = GOOD_HOBBIES[int(rng.integers(0, len(GOOD_HOBBIES)))]
    return Person(name=str(name), age=age, hobbies=list(hobbies))


## Run and score

200 trials per strategy. Metrics:

- **validity** - what fraction return a non-None Person.
- **latency cost** - number of LLM calls made (always 1 for prompt-only
  and FSM, up to 3 for validate+retry).


In [ ]:
N_TRIALS = 200

def run(strategy, N: int = N_TRIALS) -> tuple[float, int]:
    llm = LLM()
    valid = 0
    total_calls = 0
    for _ in range(N):
        before = llm.rng.bit_generator.state["state"]["state"]
        out = strategy(llm, "Generate a Person object")
        after = llm.rng.bit_generator.state["state"]["state"]
        if out is not None and isinstance(out, Person):
            valid += 1
        # Approximate calls by counting how much the rng state advanced;
        # each generate() call advances the stream similarly.
        total_calls += 1
    return valid / N, total_calls


valid_po, _ = run(strategy_prompt_only)
valid_vr, _ = run(strategy_validate_retry)
valid_fsm, _ = run(strategy_fsm_constrained)

print(f"prompt-only        validity = {valid_po:.3f}")
print(f"validate+retry     validity = {valid_vr:.3f}")
print(f"FSM-constrained    validity = {valid_fsm:.3f}")


In [ ]:
s.check(
    "prompt_only_matches_flaky_baseline",
    lambda: 0.55 <= valid_po <= 0.80,
    msg=f"validity = {valid_po:.3f}  (expected ~0.70)",
)
s.check(
    "validate_retry_lifts_validity_over_prompt_only",
    lambda: valid_vr > valid_po,
    msg=f"prompt-only={valid_po:.3f}  validate+retry={valid_vr:.3f}",
)
s.check(
    "fsm_constrained_is_perfect",
    lambda: valid_fsm == 1.0,
    msg=f"validity = {valid_fsm:.3f}",
)
s.check(
    "retry_validity_near_one",
    lambda: valid_vr >= 0.98,
    msg=f"validate+retry validity = {valid_vr:.3f}",
)
# Pydantic rejects out-of-range ages.
bad_ages_rejected = True
for a in (-1, 131, 1000):
    try:
        Person(name="x", age=a, hobbies=[])
        bad_ages_rejected = False
    except ValidationError:
        pass
s.check("pydantic_enforces_age_range", lambda: bad_ages_rejected)


### Validity vs latency - the three-way tradeoff

Two bar charts side by side: schema compliance on the left, average LLM calls per successful output on the right. Prompt-only is cheapest but leaks malformed outputs at the base rate of the model. Validate+retry climbs to near-perfect compliance but pays extra calls on failures. FSM-constrained is perfect at exactly one call per output, which is why it has become the default in production function-calling APIs.


In [ ]:
import matplotlib.pyplot as plt


class CountingLLM(LLM):
    def __init__(self):
        super().__init__(); self.calls = 0
    def generate(self, prompt, retry_correction=None):
        self.calls += 1; return super().generate(prompt, retry_correction)


def measure(strategy, N=N_TRIALS):
    llm = CountingLLM()
    ok = sum(1 for _ in range(N) if strategy(llm, "Generate a Person object") is not None)
    return ok / N, llm.calls / N


po = measure(strategy_prompt_only)
vr = measure(strategy_validate_retry)
fsm = (valid_fsm, 1.0)

names = ["prompt-only", "validate+retry", "FSM-constrained"]
validity = [po[0], vr[0], fsm[0]]
calls = [po[1], vr[1], fsm[1]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
colors = ["tab:orange", "tab:blue", "tab:green"]
ax1.bar(names, validity, color=colors)
ax1.set_ylim(0, 1.05); ax1.set_ylabel("validity (fraction)")
ax1.set_title("schema compliance across 200 trials")
for i, v in enumerate(validity):
    ax1.text(i, v + 0.02, f"{v:.2f}", ha="center")

ax2.bar(names, calls, color=colors)
ax2.set_ylabel("avg LLM calls per output")
ax2.set_title("latency cost (calls ≈ wall time)")
for i, v in enumerate(calls):
    ax2.text(i, v + 0.02, f"{v:.2f}", ha="center")
fig.tight_layout(); plt.show()


## Exercises

1. **Real Outlines.** `pip install outlines` and
   `outlines.generate.json(model, Person)`. Compare latency to
   unconstrained generation at batch=1.
2. **Cost-vs-quality curve.** For validate+retry, sweep
   `max_retries ∈ {0, 1, 2, 3, 5}` and plot validity vs total LLM
   calls per successful output.
3. **Extend the schema.** Add a nested `Address` submodel and retry
   on the real model: nested-JSON validation is where prompt-only
   tends to fall off the cliff.

## References

- Willard & Louf 2023 for the FSM-decoding algorithm.
- Pydantic v2 docs on `model_validate_json` and
  `TypeAdapter.validate_python`.
- Function-calling docs from OpenAI / Anthropic / Google - all
  three providers now offer native structured output, which is
  essentially strategy 3 pushed into the inference layer.


In [ ]:
s.summary()
s.save()
